In [0]:
-- OMNIRETAIL – PHASE 1 
-- Relational Model + Synthetic Data + Advanced SQL Analytics

-- DROP TABLES
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS stores;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS payments;
DROP TABLE IF EXISTS subscriptions;
DROP TABLE IF EXISTS event_logs;

-- CREATE TABLES (DELTA)
-- The relational schema was designed in Third Normal Form (3NF) to eliminate redundancy and update anomalies.

-- Core Entities
-- - customers
-- - stores
-- - products
-- - orders
-- - order_items
-- - subscriptions
-- - payments
-- - event_logs

-- Primary and Foreign Keys
-- - orders.customer_id → customers.customer_id
-- - orders.store_id → stores.store_id
-- - order_items.order_id → orders.order_id
-- - order_items.product_id → products.product_id
-- - payments.order_id → orders.order_id
-- This enforces referential integrity and prevents orphan records.
CREATE TABLE customers (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    state STRING,
    created_at TIMESTAMP
) USING DELTA;

CREATE TABLE stores (
    store_id STRING,
    store_name STRING,
    city STRING,
    state STRING
) USING DELTA;

CREATE TABLE products (
    product_id STRING,
    product_name STRING,
    category STRING,
    base_price DOUBLE
) USING DELTA;

CREATE TABLE orders (
    order_id STRING,
    customer_id STRING,
    store_id STRING,
    order_date DATE,
    order_status STRING
) USING DELTA;

CREATE TABLE order_items (
    order_item_id STRING,
    order_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DOUBLE
) USING DELTA;

CREATE TABLE payments (
    payment_id STRING,
    order_id STRING,
    payment_amount DOUBLE,
    payment_method STRING,
    payment_date DATE
) USING DELTA;

CREATE TABLE subscriptions (
    subscription_id STRING,
    customer_id STRING,
    start_date DATE,
    end_date DATE,
    subscription_type STRING,
    monthly_fee DOUBLE
) USING DELTA;

CREATE TABLE event_logs (
    event_id STRING,
    event_type STRING,
    customer_id STRING,
    order_id STRING,
    event_timestamp TIMESTAMP
) USING DELTA;

-- SYNTHETIC DATA GENERATION
-- Customers (200)
INSERT INTO customers
SELECT
    CONCAT('C', id),
    CONCAT('First', id),
    CONCAT('Last', id),
    CONCAT('customer', id, '@retail.com'),
    CASE 
        WHEN rand() < 0.3 THEN 'VIC'
        WHEN rand() < 0.6 THEN 'NSW'
        ELSE 'QLD'
    END,
    current_timestamp()
FROM range(1,201);

-- Stores (15)
INSERT INTO stores
SELECT
    CONCAT('S', id),
    CONCAT('Store_', id),
    CONCAT('City_', id),
    CASE 
        WHEN id % 3 = 0 THEN 'VIC'
        WHEN id % 3 = 1 THEN 'NSW'
        ELSE 'QLD'
    END
FROM range(1,16);

-- Products (100)
INSERT INTO products
SELECT
    CONCAT('P', id),
    CONCAT('Product_', id),
    CASE 
        WHEN id % 4 = 0 THEN 'Electronics'
        WHEN id % 4 = 1 THEN 'Furniture'
        WHEN id % 4 = 2 THEN 'Clothing'
        ELSE 'Groceries'
    END,
    ROUND(rand()*900 + 50,2)
FROM range(1,101);

-- Orders (3000)
INSERT INTO orders
SELECT
    CONCAT('O', id),
    CONCAT('C', CAST(rand()*199 + 1 AS INT)),
    CONCAT('S', CAST(rand()*14 + 1 AS INT)),
    DATE_ADD('2024-01-01', CAST(rand()*365 AS INT)),
    'Completed'
FROM range(1,3001);

-- Order Items (9000)
INSERT INTO order_items
SELECT
    CONCAT('OI', id),
    CONCAT('O', CAST(rand()*2999 + 1 AS INT)),
    CONCAT('P', CAST(rand()*99 + 1 AS INT)),
    CAST(rand()*4 + 1 AS INT),
    ROUND(rand()*900 + 50,2)
FROM range(1,9001);

-- Payments (1 per order)
INSERT INTO payments
SELECT
    CONCAT('PAY', id),
    CONCAT('O', id),
    ROUND(rand()*5000 + 100,2),
    CASE 
        WHEN rand() < 0.5 THEN 'Card'
        ELSE 'Cash'
    END,
    DATE_ADD('2024-01-01', CAST(rand()*365 AS INT))
FROM range(1,3001);

-- Subscriptions (100 random customers)
INSERT INTO subscriptions
SELECT
    CONCAT('SUB', id),
    CONCAT('C', id),
    DATE_ADD('2024-01-01', CAST(rand()*200 AS INT)),
    DATE_ADD('2024-01-01', CAST(rand()*365 + 200 AS INT)),
    CASE WHEN rand() < 0.5 THEN 'Premium' ELSE 'Basic' END,
    ROUND(rand()*100 + 20,2)
FROM range(1,101);

-- ADVANCED ENTERPRISE SQL ANALYTICS
-- Advanced SQL features were implemented:
-- - CTE-based modular queries
-- - Window functions (RANK, DENSE_RANK)
-- - Running totals
-- - LEFT JOIN NULL detection (products never sold)
-- - CLV calculation
-- - Revenue contribution percentage per category

-- Performance considerations included:
-- - Correct aggregation grain
-- - Avoiding unnecessary cross joins
-- - Logical partitioning in window functions

-- Total Order Value Per Order
WITH order_totals AS (
    SELECT order_id,
           SUM(quantity * unit_price) AS total_order_value
    FROM order_items
    GROUP BY order_id
)
SELECT * FROM order_totals LIMIT 10;

-- Revenue by Store and State
WITH revenue_base AS (
    SELECT s.store_id,
           s.state,
           oi.quantity * oi.unit_price AS revenue
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN stores s ON o.store_id = s.store_id
)
SELECT store_id, state, SUM(revenue) AS total_revenue
FROM revenue_base
GROUP BY store_id, state
ORDER BY total_revenue DESC
LIMIT 10;

-- Top 3 Customers Per State
WITH customer_revenue AS (
    SELECT c.customer_id,
           c.state,
           SUM(oi.quantity * oi.unit_price) AS total_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_id, c.state
),
ranked AS (
    SELECT *,
           DENSE_RANK() OVER (PARTITION BY state ORDER BY total_revenue DESC) AS rank
    FROM customer_revenue
)
SELECT * FROM ranked WHERE rank <= 3;

-- Monthly Revenue Trend + Running Total
WITH monthly_revenue AS (
    SELECT DATE_TRUNC('month', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY DATE_TRUNC('month', o.order_date)
)
SELECT *,
       SUM(revenue) OVER (ORDER BY month 
       ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM monthly_revenue;

-- Products Never Sold
SELECT p.product_id, p.product_name
FROM products p
LEFT JOIN order_items oi 
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL;

-- Customer Lifetime Value (CLV)
SELECT c.customer_id,
       SUM(oi.quantity * oi.unit_price) AS lifetime_value
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id
ORDER BY lifetime_value DESC
LIMIT 10;

-- Revenue Contribution % Per Category
WITH category_revenue AS (
    SELECT p.category,
           SUM(oi.quantity * oi.unit_price) AS revenue
    FROM products p
    JOIN order_items oi ON p.product_id = oi.product_id
    GROUP BY p.category
),
total_revenue AS (
    SELECT SUM(revenue) AS grand_total FROM category_revenue
)
SELECT cr.category,
       cr.revenue,
       ROUND((cr.revenue / tr.grand_total) * 100,2) AS contribution_percent
FROM category_revenue cr
CROSS JOIN total_revenue tr;

-- Fraud Risk Ranking (High Value Orders)
WITH order_values AS (
    SELECT o.order_id,
           SUM(oi.quantity * oi.unit_price) AS order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_id
)
SELECT *,
       RANK() OVER (ORDER BY order_value DESC) AS fraud_risk_rank
FROM order_values
ORDER BY fraud_risk_rank
LIMIT 10;


CREATE VOLUME IF NOT EXISTS omniretailx_volume;